# Amostragem

A **amostragem** permite estudar uma população a partir de um subconjunto de observações. Em ciência de dados, ela é útil tanto para **inferência estatística** quanto para reduzir custo computacional, criar conjuntos de treino e teste ou trabalhar com fluxos de dados muito grandes.

Uma boa amostra não precisa reproduzir perfeitamente a população, mas o **processo de seleção** deve evitar vieses sistemáticos e ser compatível com o objetivo da análise. Como discutido por {cite:p}`bruce2020practical`, diferentes estratégias de amostragem atendem a diferentes estruturas de dados.

Neste notebook, vamos comparar cinco abordagens usando o conjunto `census.csv`:
**aleatória simples, sistemática, por agrupamento, estratificada e de reservatório**.

In [1]:
import random

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

Antes de amostrar, fazemos uma inspeção curta da população. Isso é importante porque o tamanho do conjunto, as variáveis disponíveis e a distribuição da variável usada como estrato influenciam a escolha do método.

In [2]:
dataset = pd.read_csv('data/census.csv')

print(f'Linhas: {dataset.shape[0]:,}')
print(f'Colunas: {dataset.shape[1]}')
dataset.head()

Linhas: 32,561
Colunas: 15


,age,workclass,final-weight,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loos,hour-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## Técnicas de amostragem

### Amostragem aleatória simples (AAS)

Na **Amostragem Aleatória Simples (AAS)**, cada elemento da população tem a mesma chance de entrar na amostra. Quando selecionamos $n$ observações, sem reposição, de uma população com $N$ elementos, a probabilidade de inclusão de cada elemento é

$$
P(\text{inclusão}) = \frac{n}{N}.
$$

Assim, em uma amostra de 100 indivíduos retirada de uma população de 1.000, cada indivíduo tem probabilidade $100/1000 = 10\%$ de pertencer à amostra.

A AAS é simples e possui boas propriedades estatísticas, mas uma realização específica **não garante** que pequenos subgrupos da população sejam bem representados. Esse ponto motiva métodos como a amostragem estratificada.

In [3]:
def amostragem_aleatoria_simples(dataset, n_amostras, random_state=1):
    return dataset.sample(n=n_amostras, replace=False, random_state=random_state)

df_amostra_aleatoria_simples = amostragem_aleatoria_simples(dataset, 100)

print(df_amostra_aleatoria_simples.shape)
df_amostra_aleatoria_simples.head()

(100, 15)


,age,workclass,final-weight,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loos,hour-per-week,native-country,income
9646,62,Self-emp-not-inc,26911,7th-8th,4,Widowed,Other-service,Not-in-family,White,Female,0,0,66,United-States,<=50K
709,18,Private,208103,11th,7,Never-married,Other-service,Other-relative,White,Male,0,0,25,United-States,<=50K
7385,25,Private,102476,Bachelors,13,Never-married,Farming-fishing,Own-child,White,Male,27828,0,50,United-States,>50K
16671,33,Private,511517,HS-grad,9,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,40,United-States,<=50K
21932,36,Private,292570,11th,7,Never-married,Machine-op-inspct,Unmarried,White,Female,0,0,40,United-States,<=50K


### Amostragem sistemática

Na **amostragem sistemática**, escolhemos um ponto inicial aleatório e percorremos a população em intervalos aproximadamente regulares. Para uma população de tamanho $N$ e uma amostra de tamanho $n$, o intervalo é

$$
k = \frac{N}{n}.
\]

Se $N=1000$ e $n=100$, temos $k=10$. Um início igual a 7 produziria posições como 7, 17, 27, 37 e assim por diante.

O método é simples e distribui as observações ao longo da lista, mas exige cuidado quando a **ordem dos dados apresenta periodicidade ou algum padrão relacionado ao intervalo escolhido**.

In [4]:
def amostragem_sistematica(dataset, n_amostras, random_state=1):
    if not 0 < n_amostras <= len(dataset):
        raise ValueError('n_amostras deve estar entre 1 e o tamanho do dataset.')

    intervalo = len(dataset) / n_amostras
    rng = np.random.default_rng(random_state)
    inicio = rng.uniform(0, intervalo)

    indices = np.floor(
        inicio + np.arange(n_amostras) * intervalo
    ).astype(int)

    return dataset.iloc[indices].copy()

df_amostra_sistematica = amostragem_sistematica(dataset, 100)

print(df_amostra_sistematica.shape)
df_amostra_sistematica.head()

(100, 15)


,age,workclass,final-weight,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loos,hour-per-week,native-country,income
166,39,Federal-gov,235485,Assoc-acdm,12,Never-married,Exec-managerial,Not-in-family,White,Male,0,0,42,United-States,<=50K
492,35,Private,32220,Assoc-acdm,12,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,60,United-States,<=50K
817,56,Private,186556,Some-college,10,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,50,United-States,>50K
1143,40,Private,214010,Bachelors,13,Never-married,Other-service,Not-in-family,White,Male,0,0,37,United-States,<=50K
1469,40,Private,228535,Assoc-acdm,12,Married-civ-spouse,Prof-specialty,Husband,White,Male,7298,0,36,United-States,>50K


A diferença em relação à AAS está no mecanismo de seleção: na AAS os registros são sorteados individualmente; na sistemática, apenas o ponto inicial é aleatório e os demais registros seguem o intervalo definido.

### Amostragem por agrupamento

Na **amostragem por agrupamento**, ou por **conglomerados**, a população é dividida em grupos e um ou mais grupos são sorteados. Em uma aplicação real, esses grupos costumam ter significado natural — por exemplo, escolas, bairros ou unidades de uma empresa.

Ela não deve ser confundida com a amostragem estratificada. Na estratificação, buscamos representar **todos os estratos**; no agrupamento, selecionamos apenas **alguns grupos** e trabalhamos com as observações pertencentes a eles.

A seguir, os grupos são criados artificialmente a partir da ordem das linhas apenas para demonstrar o mecanismo. Portanto, este exemplo não representa um desenho amostral por conglomerados baseado em grupos naturais.

In [5]:
def amostragem_agrupamento(dataset, numero_grupos, random_state=1):
    if not 1 <= numero_grupos <= len(dataset):
        raise ValueError('numero_grupos deve estar entre 1 e o tamanho do dataset.')

    grupos = np.array_split(np.arange(len(dataset)), numero_grupos)
    rng = np.random.default_rng(random_state)
    grupo_selecionado = int(rng.integers(0, numero_grupos))

    amostra = dataset.iloc[grupos[grupo_selecionado]].copy()
    amostra['grupo'] = grupo_selecionado

    return amostra

# Aproximadamente 100 registros por grupo
numero_grupos = round(len(dataset) / 100)
df_amostra_agrupamento = amostragem_agrupamento(dataset, numero_grupos)

print(df_amostra_agrupamento.shape)
df_amostra_agrupamento.head()

(100, 16)


,age,workclass,final-weight,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loos,hour-per-week,native-country,income,grupo
15400,44,Private,126199,Some-college,10,Divorced,Transport-moving,Unmarried,White,Male,1831,0,50,United-States,<=50K,154
15401,26,Private,165510,HS-grad,9,Divorced,Craft-repair,Unmarried,White,Male,0,0,40,United-States,<=50K,154
15402,35,Local-gov,216068,HS-grad,9,Separated,Adm-clerical,Unmarried,White,Female,0,0,38,United-States,<=50K,154
15403,23,Private,215624,Some-college,10,Never-married,Machine-op-inspct,Unmarried,Asian-Pac-Islander,Male,0,0,40,Vietnam,<=50K,154
15404,40,Private,239708,HS-grad,9,Divorced,Craft-repair,Not-in-family,White,Male,0,0,50,United-States,<=50K,154


### Amostragem estratificada

Na **amostragem estratificada**, a população é dividida em estratos definidos por uma característica relevante e a amostra é retirada preservando a participação desses grupos.

Na versão **proporcional**, se um estrato contém 20% da população, ele também deve representar aproximadamente 20% da amostra. Esse método é especialmente útil quando queremos garantir a presença de grupos importantes que poderiam aparecer em proporções inadequadas em uma amostra puramente aleatória.

Aqui, a variável `income` será usada como estrato.

In [6]:
proporcao_renda = (
    dataset['income']
    .value_counts(normalize=True)
    .rename('proporção')
    .to_frame()
)

proporcao_renda

,proporção
income,
<=50K,0.75919
>50K,0.24081


In [7]:
def amostragem_estratificada(
    dataset,
    coluna_estrato,
    n_amostras,
    random_state=1
):
    _, amostra = train_test_split(
        dataset,
        test_size=n_amostras,
        stratify=dataset[coluna_estrato],
        random_state=random_state
    )
    return amostra.copy()

df_amostra_estratificada = amostragem_estratificada(
    dataset,
    coluna_estrato='income',
    n_amostras=100
)

print(df_amostra_estratificada.shape)
df_amostra_estratificada['income'].value_counts(normalize=True)

(100, 15)


income
<=50K    0.76
>50K     0.24
Name: proportion, dtype: float64

Observe que a estratificação não tenta preservar automaticamente todas as características da população. Neste caso, ela foi construída especificamente para preservar a distribuição de `income`.

### Amostragem de reservatório

A **amostragem de reservatório** é apropriada quando os dados chegam como um fluxo (*stream*) ou quando não queremos armazenar toda a população antes de selecionar a amostra.

Para manter um reservatório com $k$ elementos, os primeiros $k$ registros são armazenados. A partir do elemento de posição $i$, sorteamos um índice entre 0 e $i$; se esse índice estiver dentro do reservatório, o novo elemento substitui uma observação existente.

Ao final do processo, cada elemento processado possui a mesma probabilidade de pertencer à amostra, apesar de não ser necessário conhecer antecipadamente o tamanho total do fluxo.

In [8]:
def amostragem_reservatorio(dataset, n_amostras, random_state=1):
    if not 0 < n_amostras <= len(dataset):
        raise ValueError('n_amostras deve estar entre 1 e o tamanho do dataset.')

    rng = random.Random(random_state)
    reservatorio = list(range(n_amostras))

    for i in range(n_amostras, len(dataset)):
        j = rng.randint(0, i)
        if j < n_amostras:
            reservatorio[j] = i

    return dataset.iloc[reservatorio].copy()

df_amostragem_reservatorio = amostragem_reservatorio(dataset, 100)

print(df_amostragem_reservatorio.shape)
df_amostragem_reservatorio.head()

(100, 15)


,age,workclass,final-weight,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loos,hour-per-week,native-country,income
29611,47,Private,163814,10th,6,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,40,United-States,<=50K
21698,32,?,256371,12th,8,Never-married,?,Own-child,Black,Female,0,0,40,United-States,<=50K
30679,51,Private,182944,HS-grad,9,Widowed,Tech-support,Unmarried,Black,Female,0,0,40,United-States,<=50K
28553,25,Private,154941,Bachelors,13,Married-civ-spouse,Sales,Husband,White,Male,0,0,40,United-States,>50K
8772,39,Private,114678,HS-grad,9,Divorced,Other-service,Unmarried,Black,Female,5455,0,40,United-States,<=50K


A diferença central em relação à AAS é operacional: se toda a população já está disponível em memória, uma amostra aleatória simples é mais direta. O reservatório se torna interessante quando os registros chegam sequencialmente e o tamanho final do fluxo pode ser muito grande ou desconhecido.

## Comparação das amostras

Uma única estatística não é suficiente para dizer qual técnica é “melhor”, mas podemos verificar se algumas características da população foram aproximadamente preservadas. Vamos comparar o tamanho da amostra, a média de idade e a proporção de indivíduos com renda acima de 50 mil.

In [9]:
amostras = {
    'Original': dataset,
    'Aleatória simples': df_amostra_aleatoria_simples,
    'Sistemática': df_amostra_sistematica,
    'Agrupamento': df_amostra_agrupamento,
    'Estratificada': df_amostra_estratificada,
    'Reservatório': df_amostragem_reservatorio
}

def proporcao_alta_renda(df):
    renda = df['income'].astype(str).str.strip()
    return (renda == '>50K').mean() * 100

df_comparacao = pd.DataFrame([
    {
        'Método': nome,
        'n': len(df),
        'Média da idade': df['age'].mean(),
        'Renda >50K (%)': proporcao_alta_renda(df)
    }
    for nome, df in amostras.items()
])

media_populacao = df_comparacao.loc[0, 'Média da idade']
renda_populacao = df_comparacao.loc[0, 'Renda >50K (%)']

df_comparacao['Δ média idade'] = (
    df_comparacao['Média da idade'] - media_populacao
).abs()

df_comparacao['Δ renda >50K (p.p.)'] = (
    df_comparacao['Renda >50K (%)'] - renda_populacao
).abs()

df_comparacao.round(2)

,Método,n,Média da idade,Renda >50K (%),Δ média idade,Δ renda >50K (p.p.)
0,Original,32561,38.58,24.08,0.00,0.00
1,Aleatória simples,100,39.41,24.00,0.83,0.08
2,Sistemática,100,36.12,21.00,2.46,3.08
3,Agrupamento,100,38.57,28.00,0.01,3.92
4,Estratificada,100,37.24,24.00,1.34,0.08
5,Reservatório,100,39.83,27.00,1.25,2.92


A tabela deve ser interpretada com cuidado. Diferenças entre as amostras são esperadas por variabilidade amostral e **uma única realização não permite concluir qual método é superior**.

Alguns comportamentos, porém, vêm do próprio desenho: a amostra estratificada tende a preservar melhor a proporção de `income` porque essa foi justamente a variável usada como estrato. Já o resultado do agrupamento depende fortemente de como os grupos são formados.

Para comparar métodos de forma mais rigorosa, seria necessário repetir o processo muitas vezes e analisar a distribuição dos erros de cada estimativa.

**Em resumo:** use AAS quando a população está disponível e não há necessidade de controlar subgrupos; sistemática quando há uma lista ordenada sem periodicidade problemática; estratificada quando determinados grupos precisam manter representação; agrupamento quando a coleta ocorre naturalmente por grupos; e reservatório para dados em fluxo.